# IPA service-ticket trends

Reruns the service-ticket analysis on fresh exports: **unplanned tickets per 100 installed machine-months**, by
issue (request group and Category), for every reporting period, broken down by **install cohort** (machines
installed in different date ranges), product family and model. Each run compares the latest complete period with
the previous 12 months and calls out changes conservatively, and lists install cohorts that stand out.

**Each cycle**
1. About two weeks after the period ends, save the new case report and that month's equipment list into
   `trends/data/raw/` (keep the earlier equipment lists there too).
2. Check the parameters below. Normally nothing changes; to switch cadence, edit only `RERUN_FREQUENCY`
   (`'quarterly'` or `'monthly'`; any whole number of months works).
3. **Run All.** The results appear below and in `trends/data/output/latest/`: `report.pdf` to attach,
   `email_summary.txt` to paste, and CSV tables for the dashboard.

If something is wrong with an input, the run stops before computing anything and says how to fix it. See
`trends/README.md` for the full runbook and `trends/METHOD.md` for the definitions and statistics.

In [ ]:
# Parameters: edit here, or pass them with papermill (-p NAME value, or -f params.yaml).

# ---- Cadence
RERUN_FREQUENCY = 'quarterly'  # how often this is rerun = length of each period: 'quarterly', 'monthly', '2 months'...
YEAR_START_MONTH = 1  # first month of the reporting year (10 for an October fiscal year)
FISCAL_YEAR_NAMED_BY = 'end'  # 'end': the year starting Oct 2026 is FY2027
ANCHOR_DATE = None  # 'YYYY-MM-DD' a date that starts a period, for unusual calendars
PERIOD_EDGES = None  # explicit period edges ['YYYY-MM-DD', ...] (overrides RERUN_FREQUENCY)
LATEST_PERIOD = None  # report an earlier final period as the latest (back-runs); never overwrites latest/
BASELINE_MONTHS = None  # months of baseline before the latest period (default: 12, or one period if longer)
HISTORY_MONTHS = None  # months of history in the charts (default: 24, or three periods if longer)
LABEL_LAG_DAYS = 14  # days after a period ends before it counts as final (Categories are filled at close)
MAX_OPEN_SHARE = 0.02  # a period with more of its tickets still open stays provisional
MAX_BLANK_CATEGORY_SHARE = 0.05  # above this share of blank Categories, issue-level comparisons skip the period
MAX_UNMAPPED_SHARE = 0.05  # above this share of unmapped tickets, affected groups are not tested

# ---- Inputs (relative paths are read from the trends folder)
CASE_EXPORTS = 'data/raw/IPA_CaseHistory_*.xlsx'  # case report(s) from Salesforce (.xlsx); the newest match is used
CASE_SHEET = 'ipa_repair cases'  # sheet holding the cases (if missing and only one other sheet exists, that one is read)
EQUIPMENT_LISTS = 'data/raw/IPA_Active_Equipment_list_*.xlsx'  # every equipment-list snapshot (Monthly Report); keep them all
EQUIPMENT_SHEET = 'Equipment list - Monthly Report'  # sheet holding the equipment list
EQUIPMENT_LIST_DATES = {}  # {'file name': 'YYYY-MM-DD'} for lists whose file name has no date
EXTRACTED_AT = None  # 'YYYY-MM-DD' the case report was exported (default: the workbook's created date)
DATA_START = None  # 'YYYY-MM-DD' first day the export covers (default: the first ticket's day)
DATA_END = None  # 'YYYY-MM-DD' last day the export covers (default: the last ticket's day)
COLUMN_ALIASES = {}  # {'renamed header': 'expected header'} if IPA renames a column
CATEGORY_ALIASES = {}  # {'new Category name': 'old Category name'} so a renamed Category keeps its series
REQUEST_TYPES_LOCAL = None  # CSV adding new Trouble Code x Category combinations to the request groups
UNMAPPED_POLICY = 'separate'  # 'separate': unmapped combinations form their own, never-tested group
CASE_CLOCK_TZ = 'America/New_York'  # time zone of the case times in the export (US Eastern for IPA)
DATE_FORMAT = None  # e.g. '%m/%d/%Y %I:%M %p' when dates are stored as text (default: detected)
BASE_DIR = None  # folder relative paths are read from (default: the trends folder)
OUTPUT_DIR = 'data/output'  # where runs are written (runs/<run_id>/ and latest/)

# ---- Breakdowns
STRATIFY_BY = ['install_cohort', 'family', 'model']  # breakdowns: 'install_cohort', 'family', 'model', 'age_group', 'service_status', ...
CATEGORY_DIMS = ['fleet', 'install_cohort']  # breakdowns that also get Category-level rates
INSTALL_COHORT_WIDTH = '1 year'  # width of install cohorts: '1 year', '6 months', 'quarterly'...
INSTALL_COHORT_START = '2016-01-01'  # first cohort start; earlier installs are pooled ('Before 2016')
INSTALL_COHORT_EDGES = None  # explicit cohort edges ['2016-01-01', '2019-01-01', ...] (overrides the width)
MODEL_FILTER = None  # only these models (reconciliation)
POOL_SMALL_STRATA = True  # pool models with too few machines into one "Other models" series
MIN_TYPE_MONTHS = 100  # machine-months a model needs over 396 days for its own series (100 = 7.7 machines)
MIN_CELL_MONTHS = 100  # cells with fewer machine-months are shown hollow
THIRD_PARTY_PLANS = ['Vogeltronix']  # service plans whose service does not reach the case export (excluded)
MISSING_INSTALL_DATE = 'exclude'  # 'exclude' machines without an install date, or 'data_start'

# ---- Call-outs (conservative screening flags)
Q_CUT = 0.05  # false-discovery rate for a flag
WATCH_Q = 0.2  # false-discovery rate for the watch list (report only)
MIN_CHANGE = 0.2  # smallest change worth flagging (0.20 = 20%)
MIN_CONTRAST = 0.5  # smallest cohort difference worth calling out (0.50 = 50% higher)
CONTRAST_MONTHS = 12  # months of final periods in the install-cohort comparison
MIN_EXPECTED = 10  # fewest expected tickets in the latest period for a test
MIN_BASELINE_TICKETS = 20  # fewest baseline tickets for a test
MIN_ACCOUNTS = 10  # fewest accounts with baseline tickets for a test
TAU2_FLOOR = 0.001  # smallest between-period variance assumed (log scale)
N_BOOT = 2000  # bootstrap draws (2000 as in the analysis IPA saw)
SEED = 15072  # random seed (15072 as in the analysis IPA saw)
COHORT_MAP_ISSUES = 'auto'  # 'auto' or a list of issue names for the cohort charts

# ---- Outputs
MAKE_CHARTS = 'auto'  # 'auto': charts when matplotlib is installed
MAKE_REPORT = True  # write report.html, report.pdf and the email text
REPORT_TITLE = 'IPA service-ticket trends'  # title of the report and email

In [ ]:
import pathlib
import sys

_here = pathlib.Path.cwd()
for _p in (_here, _here / 'trends', _here.parent):
    if (_p / 'ipa_trends').is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from ipa_trends import config, notebook

params = config.collect(globals())   # every ALL-CAPS name above; a misspelt name stops the run
result = notebook.run_and_show(params)

## Next steps

- Read the data-quality notes above. A period marked *provisional* is reported in the next run.
- Send `latest/report.pdf` with the text of `latest/email_summary.txt` to the distribution list, once
  "Ready to distribute" says yes.
- The Product Insights Dashboard reads `latest/rates_final.csv` (and the other tables; columns in `schema.json`).
- Flags are screening flags for follow-up, not proof of a problem.